In [1]:
model_name_str = "cl-tohoku/bert-base-japanese-v2"
model_type_str = "tohoku-bb2"
batch_size_int = 1

dirpath_output_str = "scores/score_" + model_type_str + "_" + str(batch_size_int)

print(f"{model_name_str = }")
print(f"{dirpath_output_str = }")

model_name_str = 'cl-tohoku/bert-base-japanese-v2'
dirpath_output_str = 'scores/score_tohoku-bb2_1'


In [2]:
import json

filepath_dataset_eval_str = "../datasets/jsts-v1.1/valid-v1.1.json"

label_str = "label"
sentence1_str = "sentence1"
sentence2_str = "sentence2"


data_eval_dict = dict()
labels_list = list()
sentences1_list = list()
sentences2_list = list()

with open(filepath_dataset_eval_str) as fp:
    for line in fp:
        line_dict = json.loads(line)
        labels_list.append(line_dict[label_str])
        sentences1_list.append(line_dict[sentence1_str])
        sentences2_list.append(line_dict[sentence2_str])

data_eval_dict[label_str] = labels_list
data_eval_dict[sentence1_str] = sentences1_list
data_eval_dict[sentence2_str] = sentences2_list

# print(data_eval_dict)

In [ ]:
from sentence_transformers import SentenceTransformer, models


transformer = models.Transformer(model_name_str)

pooling = models.Pooling(transformer.get_word_embedding_dimension(), pooling_mode='mean')
model = SentenceTransformer(modules=[transformer, pooling])

In [4]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction

evaluator = EmbeddingSimilarityEvaluator(
    sentences1=data_eval_dict[sentence1_str],
    sentences2=data_eval_dict[sentence2_str],
    scores=data_eval_dict[label_str],
    batch_size=batch_size_int,  # 16
    # main_similarity=None,  # Optional[sentence_transformers.evaluation.SimilarityFunction.SimilarityFunction]
    name="",
    show_progress_bar=True,  # False
    write_csv=True
)
model.evaluate(evaluator=evaluator, output_path=dirpath_output_str)

Batches:   0%|          | 0/1457 [00:00<?, ?it/s]

Batches:   0%|          | 0/1457 [00:00<?, ?it/s]

0.7066926787664243

In [5]:
from sentence_transformers import util
from scipy.stats import pearsonr, spearmanr


embeddings1_list = model.encode(data_eval_dict[sentence1_str])
embeddings2_list = model.encode(data_eval_dict[sentence2_str])

cossim_list = list()
for embedding1, embedding2 in zip(embeddings1_list, embeddings2_list):
    cossim_float = util.cos_sim(embedding1, embedding2).tolist()[0][0]
    cossim_list.append(cossim_float)


pearson_cossim_float, _ = pearsonr(data_eval_dict[label_str], cossim_list)
spearman_cossim_float, _ = spearmanr(data_eval_dict[label_str], cossim_list)

print(f"{pearson_cossim_float = }")
print(f"{spearman_cossim_float = }")


filepath_output_str = dirpath_output_str + "/manual.txt"

with open(filepath_output_str, "w") as fp:
    fp.write(f"pearson_cossim: {pearson_cossim_float}\n")
    fp.write(f"spearman_cossim: {spearman_cossim_float}\n")

pearson_cossim_float = 0.7252545022547437
spearman_cossim_float = 0.7066692193446502
